In [ ]:
import gymnasium as gym
import numpy as np

In [2]:
import dynamicauc
from dynamicauc.dataclasses import Scenario
from dynamicauc.policy import FiPlaPolicy
from dynamicauc.primitives import *

# I. Dummy Environment Initialization

In [3]:
user_features = np.array([0.5, 0.8, 0.02], dtype=np.float32)

# Setup Primitives
scenario = Scenario(
    auction=FirstPriceAuction(),
    competition=UniformCompetition(low=0.0, high=20.0),
    arrival=PiecewiseConstantPoisson(breakpoints=[5.0], intensities=[0.1,1.2]),
    kernel=SaturationKernel(base_rate=0.5, saturation_scale=10.0,delay_rate=0.1),
    display_sampler=CatalogueSampler(creatives=[[1.0, 0.0], [0.0, 1.0]]),
    exogenous=TrivialProcess(),
    policy=FiPlaPolicy(),
    horizon=10.0,
    user_features=user_features
)

In [7]:

# Create the environment
env = gym.make(
    "DynamicAuction-v0", 
    scenario=scenario, 
    stochastic_reward=True
)
print("Environment observation space:", env.observation_space.keys())

Environment observation space: KeysView({'ctx': Box(-inf, inf, (1,), float32), 'display_features': Box(-inf, inf, (2,), float32), 'horizon': Box(0.0, inf, (1,), float32), 'step': Discrete(100000), 't': Box(0.0, 10.0, (1,), float32), 'user_features': Box(-inf, inf, (3,), float32), 'y_now': Box(-inf, inf, (1,), float32)})


In [8]:
obs, info = env.reset(seed=42)

print("Env time auction_times:", info["auction_times"])

print("Initial observation: \n - ctx :", obs["ctx"])
print("\n - step_idx :", obs["step"])
print("\n - y_func(t) :", obs["y_now"])
print("\n - display_features :", obs["display_features"])
print("\n - user_features :", obs["user_features"])
print("\n - horizon :", obs["horizon"])
print("\n - current_time :", obs["t"])

Env time auction_times: [4.2929896  6.13619361 6.85399012 7.21707099 8.2193256  9.11380807
 9.63382494]
Initial observation: 
 - ctx : [0.]

 - step_idx : 0

 - y_func(t) : [0.]

 - display_features : [1. 0.]

 - user_features : [0.5  0.8  0.02]

 - horizon : [10.]

 - current_time : [4.2929897]


In [9]:
bid = scenario.policy.bid(u=obs["user_features"], x=obs["display_features"], t=obs["t"].item(), ctx=obs["ctx"].item(), y_func=env.unwrapped.state.y_func, scenario=scenario)
obs, net_reward, done, truncated, info = env.step(bid)
print("\nAfter one step: \n - net_reward :", net_reward)
print(" - done :", done)
print(" - truncated :", truncated)
print(" - info :", info)
print("\n - ctx :", obs["ctx"])
print("\n - step_idx :", obs["step"])
print("\n - y_func(t) :", obs["y_now"])
print("\n - display_features :", obs["display_features"])
print("\n - user_features :", obs["user_features"])
print("\n - horizon :", obs["horizon"])
print("\n - current_time :", obs["t"])


After one step: 
 - net_reward : 0.0
 - done : False
 - truncated : False
 - info : {'auction': {'step': np.float64(4.292989599556912), 'auction_id': 0, 'time': np.float64(4.292989599556912), 'bid': 0.21743543546343552, 'competing_bid': 1.2763451220835065, 'won': False, 'payment': 0.0, 'context': 0.0, 'y': 0.0}, 'conversions_count': 0, 'actual_conversions': [], 'is_stochastic': True}

 - ctx : [0.]

 - step_idx : 1

 - y_func(t) : [0.]

 - display_features : [0. 1.]

 - user_features : [0.5  0.8  0.02]

 - horizon : [10.]

 - current_time : [6.1361938]
